In [ ]:
# ===================================================================
# COMPLETE TROCR TRAINING WITH LINE-WISE CER - SINGLE CONFIGURATION
# Copy each cell into Colab and run sequentially
# ===================================================================

# ===================================================================
# CELL 1: Mount Google Drive
# ===================================================================
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
import io

print("\n📤 Upload the 3 pre-split JSON files:")
print("   - ocr_train.json")
print("   - ocr_val.json")
print("   - ocr_test.json")

uploaded = files.upload()

for filename in uploaded.keys():
    with open(f"/content/{filename}", 'wb') as f:
        f.write(uploaded[filename])
    print(f"✅ Saved: {filename}")

print("\n✅ Upload verification:")
import os
for json_file in ['ocr_train.json', 'ocr_val.json', 'ocr_test.json']:
    if os.path.exists(f'/content/{json_file}'):
        size = os.path.getsize(f'/content/{json_file}')
        print(f"   ✅ {json_file} ({size} bytes)")
    else:
        print(f"   ❌ {json_file} NOT FOUND")

print("\n✅ All files uploaded successfully!")

Mounted at /content/drive

📤 Upload the 3 pre-split JSON files:
   - ocr_train.json
   - ocr_val.json
   - ocr_test.json


Saving ocr_val.json to ocr_val.json
Saving ocr_train.json to ocr_train.json
Saving ocr_test.json to ocr_test.json
✅ Saved: ocr_val.json
✅ Saved: ocr_train.json
✅ Saved: ocr_test.json

✅ Upload verification:
   ✅ ocr_train.json (314832 bytes)
   ✅ ocr_val.json (75521 bytes)
   ✅ ocr_test.json (85790 bytes)

✅ All files uploaded successfully!


In [ ]:



# ===================================================================
# CELL 2: Install Dependencies & Configuration
# ===================================================================
!pip install -q transformers datasets accelerate pillow editdistance jiwer opencv-python matplotlib seaborn scipy pandas

import os
import json
import random
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Configuration
DRIVE_BASE = "/content/drive/MyDrive"
IMG_DIR = Path(f"{DRIVE_BASE}/OMR/images/Dataset cortona")
TRAIN_JSON = Path("/content/ocr_train.json")
VAL_JSON = Path("/content/ocr_val.json")
TEST_JSON = Path("/content/ocr_test.json")
OUTPUT_BASE = Path("/content/trocr_ablation_results")

OUTPUT_BASE.mkdir(exist_ok=True)

# ===================================================================
# 🎯 SELECT WHICH CONFIGURATION TO RUN
# ===================================================================
# CHANGE THESE VALUES to select which configuration you want to run

SELECTED_CONFIG = {
    "name": "enc_3_dec_0",      # Options: enc_0_dec_0, enc_3_dec_0, enc_6_dec_0, enc_9_dec_0, enc_12_dec_0, enc_0_dec_6
    "freeze_encoder": 3,         # Options: 0, 3, 6, 9, 12
    "freeze_decoder": 0          # Options: 0, 6
}

# Single seed for reproducibility
SEED = 42
USE_CLAHE = True
USE_AUG = True

# ===================================================================
# ONE-CYCLE LR SCHEDULER HYPERPARAMETERS
# ===================================================================
USE_ONECYCLE = True
ONECYCLE_MAX_LR = 5.5e-6
ONECYCLE_PCT_START = 0.1
ONECYCLE_BASE_MOMENTUM = 0.85
ONECYCLE_MAX_MOMENTUM = 0.95
ONECYCLE_INITIAL_LR = 1e-9
ONECYCLE_FINAL_DIV_FACTOR = 2.2e4

# Training hyperparameters
NUM_EPOCHS = 50
TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 2
LR = ONECYCLE_MAX_LR if USE_ONECYCLE else 3e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("SELECTED CONFIGURATION")
print("=" * 80)
print(f"Configuration: {SELECTED_CONFIG['name']}")
print(f"Encoder: {SELECTED_CONFIG['freeze_encoder']}/12 frozen")
print(f"Decoder: {SELECTED_CONFIG['freeze_decoder']}/6 frozen")
print(f"\n🚀 Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\n📊 Training Configuration:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Using One-Cycle LR: {USE_ONECYCLE}")
if USE_ONECYCLE:
    print(f"   Max LR: {ONECYCLE_MAX_LR}")
    print(f"   Initial LR: {ONECYCLE_INITIAL_LR}")
    print(f"   Warmup fraction: {ONECYCLE_PCT_START}")
print(f"   Batch size: {TRAIN_BATCH}")
print(f"   CLAHE: {USE_CLAHE}")
print(f"   Augmentation: {USE_AUG}")
print("=" * 80)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.1 MB/s eta 0:00:00
🎯 SELECTED CONFIGURATION
Configuration: enc_3_dec_0
Encoder: 3/12 frozen
Decoder: 0/6 frozen

🚀 Using device: cuda
   GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.5 GB

📊 Training Configuration:
   Epochs: 50
   Using One-Cycle LR: True
   Max LR: 5.5e-06
   Initial LR: 1e-09
   Warmup fraction: 0.1
   Batch size: 8
   CLAHE: True
   Augmentation: True


In [ ]:

# ===================================================================
# CELL 3: Load Data
# ===================================================================
print("\n📂 Loading pre-split data...")

for json_path in [TRAIN_JSON, VAL_JSON, TEST_JSON]:
    if not json_path.exists():
        raise FileNotFoundError(f"❌ Missing file: {json_path}")

with open(TRAIN_JSON, 'r') as f:
    train_data = json.load(f)
with open(VAL_JSON, 'r') as f:
    val_data = json.load(f)
with open(TEST_JSON, 'r') as f:
    test_data = json.load(f)

train_anns = train_data['annotations']
val_anns = val_data['annotations']
test_anns = test_data['annotations']

image_map = {img['id']: img for img in train_data['images']}
image_map.update({img['id']: img for img in val_data['images']})
image_map.update({img['id']: img for img in test_data['images']})

print(f"✅ Loaded:")
print(f"   Train: {len(train_anns)} lines")
print(f"   Val:   {len(val_anns)} lines")
print(f"   Test:  {len(test_anns)} lines")
print(f"   Images: {len(image_map)} unique")


📂 Loading pre-split data...
✅ Loaded:
   Train: 1666 lines
   Val:   397 lines
   Test:  454 lines
   Images: 259 unique


In [ ]:





# ===================================================================
# CELL 4: Data Augmentation
# ===================================================================
class ManuscriptAugmentation:
    def __call__(self, image: Image.Image) -> Image.Image:
        img_np = np.array(image)

        if random.random() < 0.4:
            angle = random.uniform(-5, 5)
            h, w = img_np.shape[:2]
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            img_np = cv2.warpAffine(img_np, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

        if random.random() < 0.3:
            scale = random.uniform(0.95, 1.05)
            h, w = img_np.shape[:2]
            new_h, new_w = int(h * scale), int(w * scale)
            img_np = cv2.resize(img_np, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
            if scale > 1:
                start_y = (new_h - h) // 2
                start_x = (new_w - w) // 2
                img_np = img_np[start_y:start_y+h, start_x:start_x+w]
            else:
                pad_y = (h - new_h) // 2
                pad_x = (w - new_w) // 2
                img_np = cv2.copyMakeBorder(img_np, pad_y, h-new_h-pad_y, pad_x, w-new_w-pad_x, cv2.BORDER_REPLICATE)

        if random.random() < 0.3:
            factor = random.uniform(0.8, 1.2)
            img_np = np.clip(img_np.astype(np.float32) * factor, 0, 255).astype(np.uint8)

        if random.random() < 0.3:
            factor = random.uniform(0.9, 1.1)
            mean = img_np.mean()
            img_np = np.clip((img_np - mean) * factor + mean, 0, 255).astype(np.uint8)

        return Image.fromarray(img_np)

print("✅ ManuscriptAugmentation class defined")

✅ ManuscriptAugmentation class defined


In [ ]:



# ===================================================================
# CELL 5: Dataset, Metrics, Collator
# ===================================================================
from transformers import TrOCRProcessor
import editdistance
from jiwer import wer as jiwer_wer

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

class TrOCRDataset(torch.utils.data.Dataset):
    def __init__(self, annotations, image_map, img_dir, processor,
                 max_target_length=128, augment_transform=None, use_clahe=True):
        self.annotations = annotations
        self.image_map = image_map
        self.img_dir = img_dir
        self.processor = processor
        self.max_target_length = max_target_length
        self.augment_transform = augment_transform
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.annotations)

    def apply_clahe(self, img_np):
        if len(img_np.shape) == 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(img_np)
        return Image.fromarray(enhanced).convert("RGB")

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_info = self.image_map.get(ann["image_id"])
        if img_info is None:
            raise ValueError(f"Image ID {ann['image_id']} not found")

        img_path = self.img_dir / img_info["file_name"]
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")

        image = Image.open(img_path).convert("RGB")
        x, y, w, h = ann["bbox"]
        crop = image.crop((x, y, x+w, y+h))

        if self.use_clahe:
            crop = self.apply_clahe(np.array(crop))

        if self.augment_transform is not None:
            crop = self.augment_transform(crop)

        encoding = self.processor(crop, return_tensors="pt", padding="max_length", truncation=True)
        pixel_values = encoding["pixel_values"].squeeze(0)

        text = ann.get("description", "")
        labels = self.processor.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_target_length, return_tensors="pt"
        )["input_ids"].squeeze(0)

        return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = np.mean([editdistance.eval(p, l) / max(len(l), 1) for p, l in zip(pred_str, label_str)])
    wer = jiwer_wer(label_str, pred_str)

    return {"cer": cer, "wer": wer}

class FixedDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        labels = torch.stack([item["labels"] for item in batch])
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

print("✅ Dataset, metrics, and collator defined")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Dataset, metrics, and collator defined


In [ ]:
# ===================================================================
# CELL 6: Layer Freezing Configuration (WITH DIAGNOSTICS)
# ===================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def configure_freezing(model, freeze_encoder, freeze_decoder, verbose=True):
    # DEBUG: Print what we're accessing
    print("\n🔍 DEBUG INFO:")
    print(f"  model.encoder.encoder.layer type: {type(model.encoder.encoder.layer)}")
    print(f"  model.encoder.encoder.layer length: {len(model.encoder.encoder.layer)}")
    print(f"  model.decoder.model.decoder.layers type: {type(model.decoder.model.decoder.layers)}")
    print(f"  model.decoder.model.decoder.layers length: {len(model.decoder.model.decoder.layers)}")

    num_encoder_layers = len(model.encoder.encoder.layer)
    num_decoder_layers = len(model.decoder.model.decoder.layers)

    print(f"\n✅ Detected layers:")
    print(f"  Encoder layers: {num_encoder_layers}")
    print(f"  Decoder layers: {num_decoder_layers}")

    # Freeze encoder layers
    for i, layer in enumerate(model.encoder.encoder.layer):
        for param in layer.parameters():
            param.requires_grad = (i >= freeze_encoder)

    # Freeze decoder layers
    for i, layer in enumerate(model.decoder.model.decoder.layers):
        for param in layer.parameters():
            param.requires_grad = (i >= freeze_decoder)

    # Count parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
    encoder_total = sum(p.numel() for p in model.encoder.parameters())
    decoder_trainable = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
    decoder_total = sum(p.numel() for p in model.decoder.parameters())

    config = {
        "freeze_encoder": freeze_encoder,
        "freeze_decoder": freeze_decoder,
        "total_encoder_layers": num_encoder_layers,
        "total_decoder_layers": num_decoder_layers,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_ratio": trainable / total,
        "encoder_trainable_ratio": encoder_trainable / encoder_total if encoder_total > 0 else 0,
        "decoder_trainable_ratio": decoder_trainable / decoder_total if decoder_total > 0 else 0
    }

    if verbose:
        enc_frozen_count = freeze_encoder
        enc_trainable_count = num_encoder_layers - freeze_encoder
        dec_frozen_count = freeze_decoder
        dec_trainable_count = num_decoder_layers - freeze_decoder

        enc_trainable_pct = 100 * enc_trainable_count / num_encoder_layers
        dec_trainable_pct = 100 * dec_trainable_count / num_decoder_layers

        print(f"\n{'='*70}")
        print(f"Freezing Configuration:")
        print(f"  Encoder: {enc_frozen_count}/{num_encoder_layers} frozen ({enc_trainable_pct:.0f}% trainable)")
        print(f"  Decoder: {dec_frozen_count}/{num_decoder_layers} frozen ({dec_trainable_pct:.0f}% trainable)")
        print(f"  Total trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)")
        print(f"{'='*70}\n")

    return config

print("✅ Layer freezing functions defined (with diagnostics)")

✅ Layer freezing functions defined (with diagnostics)


In [ ]:



# ===================================================================
# CELL 6: Layer Freezing Configuration
# ===================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def configure_freezing(model, freeze_encoder, freeze_decoder, verbose=True):
    num_encoder_layers = len(model.encoder.encoder.layer)
    num_decoder_layers = len(model.decoder.model.decoder.layers)

    # Freeze encoder layers
    for i, layer in enumerate(model.encoder.encoder.layer):
        for param in layer.parameters():
            param.requires_grad = (i >= freeze_encoder)

    # Freeze decoder layers
    for i, layer in enumerate(model.decoder.model.decoder.layers):
        for param in layer.parameters():
            param.requires_grad = (i >= freeze_decoder)

    # Count parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
    encoder_total = sum(p.numel() for p in model.encoder.parameters())
    decoder_trainable = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
    decoder_total = sum(p.numel() for p in model.decoder.parameters())

    config = {
        "freeze_encoder": freeze_encoder,
        "freeze_decoder": freeze_decoder,
        "total_encoder_layers": num_encoder_layers,
        "total_decoder_layers": num_decoder_layers,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_ratio": trainable / total,
        "encoder_trainable_ratio": encoder_trainable / encoder_total if encoder_total > 0 else 0,
        "decoder_trainable_ratio": decoder_trainable / decoder_total if decoder_total > 0 else 0
    }

    if verbose:
        enc_frozen_count = freeze_encoder
        enc_trainable_count = num_encoder_layers - freeze_encoder
        dec_frozen_count = freeze_decoder
        dec_trainable_count = num_decoder_layers - freeze_decoder

        enc_trainable_pct = 100 * enc_trainable_count / num_encoder_layers
        dec_trainable_pct = 100 * dec_trainable_count / num_decoder_layers

        print(f"\n{'='*70}")
        print(f"Freezing Configuration:")
        print(f"  Encoder: {enc_frozen_count}/{num_encoder_layers} frozen ({enc_trainable_pct:.0f}% trainable)")
        print(f"  Decoder: {dec_frozen_count}/{num_decoder_layers} frozen ({dec_trainable_pct:.0f}% trainable)")
        print(f"  Total trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)")
        print(f"{'='*70}\n")

    return config

print("✅ Layer freezing functions defined")

✅ Layer freezing functions defined


In [ ]:


# ===================================================================
# CELL 7: One-Cycle Trainer & Main Training Function
# ===================================================================
from transformers import VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import EarlyStoppingCallback


class _HuttnerOneCycleTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer with AdamW + OneCycleLR (paper-style scheduling)."""

    def __init__(
        self,
        *args,
        onecycle_max_lr: float,
        onecycle_pct_start: float,
        onecycle_base_momentum: float,
        onecycle_max_momentum: float,
        onecycle_initial_lr: float,
        onecycle_final_div_factor: float,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self._onecycle_max_lr = float(onecycle_max_lr)
        self._onecycle_pct_start = float(onecycle_pct_start)
        self._onecycle_base_momentum = float(onecycle_base_momentum)
        self._onecycle_max_momentum = float(onecycle_max_momentum)
        self._onecycle_initial_lr = float(onecycle_initial_lr)
        self._onecycle_final_div_factor = float(onecycle_final_div_factor)

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer

        wd = 1e-4
        params = [p for p in self.model.parameters() if p.requires_grad]

        self.optimizer = torch.optim.AdamW(
            params,
            lr=self._onecycle_max_lr,
            weight_decay=wd,
        )
        return self.optimizer

    def create_scheduler(self, num_training_steps: int, optimizer=None):
        if self.lr_scheduler is not None:
            return self.lr_scheduler

        optimizer = optimizer if optimizer is not None else self.optimizer
        if optimizer is None:
            raise RuntimeError("Optimizer must be created before scheduler")

        if self._onecycle_initial_lr <= 0:
            raise ValueError("onecycle_initial_lr must be > 0")
        div_factor = self._onecycle_max_lr / self._onecycle_initial_lr

        self.lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=self._onecycle_max_lr,
            total_steps=num_training_steps,
            pct_start=self._onecycle_pct_start,
            anneal_strategy="cos",
            cycle_momentum=True,
            base_momentum=self._onecycle_base_momentum,
            max_momentum=self._onecycle_max_momentum,
            div_factor=div_factor,
            final_div_factor=self._onecycle_final_div_factor,
        )
        return self.lr_scheduler


def run_single_experiment(exp_config, seed=SEED, use_clahe=True, use_aug=True):
    set_seed(seed)

    exp_name = f"{exp_config['name']}"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = OUTPUT_BASE / f"{exp_name}_{timestamp}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'#'*70}")
    print(f"# Starting: {exp_name}")
    print(f"# Encoder: {exp_config['freeze_encoder']}/12 frozen")
    print(f"# Decoder: {exp_config['freeze_decoder']}/6 frozen")
    print(f"# Using One-Cycle LR: {USE_ONECYCLE}")
    print(f"{'#'*70}\n")

    # Load model
    model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
    model.to(device)

    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id

    # Configure freezing
    config = configure_freezing(
        model,
        freeze_encoder=exp_config['freeze_encoder'],
        freeze_decoder=exp_config['freeze_decoder'],
        verbose=True
    )

    config.update(
        {
            "experiment_name": exp_config['name'],
            "seed": seed,
            "use_clahe": use_clahe,
            "use_aug": use_aug,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LR,
            "use_onecycle": USE_ONECYCLE,
            "onecycle_max_lr": ONECYCLE_MAX_LR if USE_ONECYCLE else None,
            "onecycle_pct_start": ONECYCLE_PCT_START if USE_ONECYCLE else None,
        }
    )

    with open(out_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    # Prepare datasets
    aug = ManuscriptAugmentation() if use_aug else None

    train_dataset = TrOCRDataset(
        train_anns, image_map, IMG_DIR, processor,
        max_target_length=128, augment_transform=aug, use_clahe=use_clahe
    )

    val_dataset = TrOCRDataset(
        val_anns, image_map, IMG_DIR, processor,
        max_target_length=128, augment_transform=None, use_clahe=use_clahe
    )

    test_dataset = TrOCRDataset(
        test_anns, image_map, IMG_DIR, processor,
        max_target_length=128, augment_transform=None, use_clahe=use_clahe
    )

    # Training arguments
    if USE_ONECYCLE:
        training_args = Seq2SeqTrainingArguments(
            output_dir=str(out_dir / "checkpoints"),
            predict_with_generate=True,
            generation_max_length=128,
            generation_num_beams=4,
            eval_strategy="epoch",
            save_strategy="epoch",
            per_device_train_batch_size=TRAIN_BATCH,
            per_device_eval_batch_size=EVAL_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=NUM_EPOCHS,
            learning_rate=ONECYCLE_MAX_LR,
            warmup_ratio=0.0,
            weight_decay=0.0,
            label_smoothing_factor=0.0,
            fp16=False,
            load_best_model_at_end=True,
            metric_for_best_model="cer",
            greater_is_better=False,
            logging_strategy="steps",
            logging_steps=50,
            save_total_limit=2,
            dataloader_num_workers=4,
            report_to="none",
            seed=seed
        )
    else:
        training_args = Seq2SeqTrainingArguments(
            output_dir=str(out_dir / "checkpoints"),
            predict_with_generate=True,
            eval_strategy="epoch",
            save_strategy="epoch",
            per_device_train_batch_size=TRAIN_BATCH,
            per_device_eval_batch_size=EVAL_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=NUM_EPOCHS,
            learning_rate=LR,
            warmup_ratio=0.1,
            weight_decay=0.01,
            label_smoothing_factor=0.1,
            fp16=True,
            load_best_model_at_end=True,
            metric_for_best_model="cer",
            greater_is_better=False,
            logging_strategy="steps",
            logging_steps=50,
            save_total_limit=2,
            dataloader_num_workers=4,
            report_to="none",
            seed=seed
        )

    # Create trainer
    if USE_ONECYCLE:
        model.generation_config.max_length = 128
        model.generation_config.num_beams = 4
        model.generation_config.early_stopping = True
        model.generation_config.no_repeat_ngram_size = 0
        model.generation_config.length_penalty = 1.0

        trainer = _HuttnerOneCycleTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=FixedDataCollator(processor),
            compute_metrics=compute_metrics,
            callbacks=[],
            onecycle_max_lr=ONECYCLE_MAX_LR,
            onecycle_pct_start=ONECYCLE_PCT_START,
            onecycle_base_momentum=ONECYCLE_BASE_MOMENTUM,
            onecycle_max_momentum=ONECYCLE_MAX_MOMENTUM,
            onecycle_initial_lr=ONECYCLE_INITIAL_LR,
            onecycle_final_div_factor=ONECYCLE_FINAL_DIV_FACTOR,
        )
    else:
        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=FixedDataCollator(processor),
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
        )

    # Train
    print("\n🚀 Starting training...")
    start_time = time.time()
    train_result = trainer.train()
    training_time = time.time() - start_time

    # Evaluate on test set
    print("\n📊 Evaluating on test set...")
    test_result = trainer.predict(test_dataset)

    # Compile results
    results = {
        "experiment": exp_name,
        "freeze_encoder": exp_config['freeze_encoder'],
        "freeze_decoder": exp_config['freeze_decoder'],
        "seed": seed,
        "config": config,
        "train_metrics": {k: float(v) if isinstance(v, (int, float, np.number)) else v
                         for k, v in train_result.metrics.items()},
        "test_metrics": {k: float(v) if isinstance(v, (int, float, np.number)) else v
                        for k, v in test_result.metrics.items()},
        "training_time_seconds": training_time,
        "output_dir": str(out_dir)
    }

    # Save results
    with open(out_dir / "results.json", "w") as f:
        json.dump(results, f, indent=2)

    # Save model
    model_dir = out_dir / "final_model"
    trainer.save_model(model_dir)
    processor.save_pretrained(model_dir)

    print(f"\n✅ Results for {exp_name}:")
    print(f"   Test CER: {test_result.metrics['test_cer']:.4f}")
    print(f"   Test WER: {test_result.metrics['test_wer']:.4f}")
    print(f"   Training time: {training_time/60:.1f} minutes")
    print(f"   Saved to: {out_dir}")

    # Clean up
    del model, trainer, train_dataset, val_dataset, test_dataset
    torch.cuda.empty_cache()

    return results

print("✅ One-Cycle Trainer and training function defined")

✅ One-Cycle Trainer and training function defined


In [ ]:




# ===================================================================
# CELL 8: RUN SELECTED CONFIGURATION
# ===================================================================
print("\n" + "="*80)
print("🎯 RUNNING SELECTED CONFIGURATION")
print("="*80)

result = run_single_experiment(
    exp_config=SELECTED_CONFIG,
    seed=SEED,
    use_clahe=USE_CLAHE,
    use_aug=USE_AUG
)

print(f"\n✅ Training complete for {SELECTED_CONFIG['name']}")
print(f"   Test CER: {result['test_metrics']['test_cer']:.4f}")
print(f"   Test WER: {result['test_metrics']['test_wer']:.4f}")


🎯 RUNNING SELECTED CONFIGURATION

######################################################################
# Starting: enc_3_dec_0
# Encoder: 3/12 frozen
# Decoder: 0/6 frozen
# Using One-Cycle LR: True
######################################################################



model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].



📊 Evaluating on test set...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Results for enc_3_dec_0:
   Test CER: 0.0823
   Test WER: 0.3238
   Training time: 131.8 minutes
   Saved to: /content/trocr_ablation_results/enc_3_dec_0_20260206_194608

✅ Training complete for enc_3_dec_0
   Test CER: 0.0823
   Test WER: 0.3238


In [ ]:



# ===================================================================
# CELL 9: COMPUTE LINE-WISE CER FOR EACH TEST SAMPLE
# ===================================================================
print("\n" + "="*80)
print("📊 COMPUTING LINE-WISE CER FOR EACH TEST SAMPLE")
print("="*80)

# Find the output directory from the result
out_dir = Path(result['output_dir'])
model_dir = out_dir / "final_model"

print(f"\n   Loading model from: {model_dir}")

# Load the trained model
from transformers import VisionEncoderDecoderModel, TrOCRProcessor

model = VisionEncoderDecoderModel.from_pretrained(model_dir)
processor_reload = TrOCRProcessor.from_pretrained(model_dir)
model.to(device)
model.eval()

# Prepare test dataset (no augmentation, with CLAHE)
test_dataset = TrOCRDataset(
    test_anns, image_map, IMG_DIR, processor_reload,
    max_target_length=128, augment_transform=None, use_clahe=USE_CLAHE
)

print(f"   Test dataset size: {len(test_dataset)} samples")

# Get predictions and compute per-line CER
from tqdm import tqdm

line_wise_results = []

print(f"\n🔍 Generating predictions for each test sample...")

for idx in tqdm(range(len(test_dataset)), desc="Processing"):
    # Get the sample
    sample = test_dataset[idx]
    ann = test_anns[idx]

    # Get ground truth
    ground_truth = ann.get("description", "")

    # Prepare input
    pixel_values = sample["pixel_values"].unsqueeze(0).to(device)

    # Generate prediction
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    # Decode prediction
    prediction = processor_reload.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Compute CER for this line
    if len(ground_truth) == 0:
        line_cer = 0.0 if len(prediction) == 0 else 1.0
    else:
        edit_dist = editdistance.eval(prediction, ground_truth)
        line_cer = edit_dist / len(ground_truth)

    # Store results
    line_result = {
        "sample_idx": idx,
        "image_id": ann["image_id"],
        "bbox": ann["bbox"],
        "ground_truth": ground_truth,
        "prediction": prediction,
        "cer": line_cer,
        "edit_distance": editdistance.eval(prediction, ground_truth) if len(ground_truth) > 0 else 0,
        "gt_length": len(ground_truth),
        "pred_length": len(prediction)
    }

    line_wise_results.append(line_result)

# Compute statistics
total_cer = sum(r["cer"] for r in line_wise_results) / len(line_wise_results)
total_edit_dist = sum(r["edit_distance"] for r in line_wise_results)
total_gt_chars = sum(r["gt_length"] for r in line_wise_results)
overall_cer = total_edit_dist / total_gt_chars if total_gt_chars > 0 else 0

print(f"\n📈 LINE-WISE CER STATISTICS:")
print(f"=" * 80)
print(f"Total test samples: {len(line_wise_results)}")
print(f"Average per-line CER: {total_cer:.4f}")
print(f"Overall CER (total edit distance / total chars): {overall_cer:.4f}")
print(f"Min CER: {min(r['cer'] for r in line_wise_results):.4f}")
print(f"Max CER: {max(r['cer'] for r in line_wise_results):.4f}")
print(f"Median CER: {sorted([r['cer'] for r in line_wise_results])[len(line_wise_results)//2]:.4f}")
print(f"=" * 80)

# Save detailed line-wise results
linewise_output_file = out_dir / "linewise_cer_results.json"

output_data = {
    "configuration": SELECTED_CONFIG,
    "statistics": {
        "total_samples": len(line_wise_results),
        "average_per_line_cer": total_cer,
        "overall_cer": overall_cer,
        "min_cer": min(r['cer'] for r in line_wise_results),
        "max_cer": max(r['cer'] for r in line_wise_results),
        "median_cer": sorted([r['cer'] for r in line_wise_results])[len(line_wise_results)//2],
        "total_edit_distance": total_edit_dist,
        "total_ground_truth_chars": total_gt_chars
    },
    "per_line_results": line_wise_results
}

with open(linewise_output_file, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"\n✅ Line-wise CER results saved to:")
print(f"   {linewise_output_file}")

# Also save a simplified CSV for easy analysis
import pandas as pd

df = pd.DataFrame(line_wise_results)
csv_file = out_dir / "linewise_cer_results.csv"
df.to_csv(csv_file, index=False, encoding="utf-8")

print(f"\n✅ CSV file saved to:")
print(f"   {csv_file}")

# Display sample predictions
print(f"\n📝 SAMPLE PREDICTIONS (first 10):")
print("=" * 80)

for i, res in enumerate(line_wise_results[:10], 1):
    print(f"\nSample {i}:")
    print(f"  Ground Truth: {res['ground_truth']}")
    print(f"  Prediction:   {res['prediction']}")
    print(f"  CER:          {res['cer']:.4f}")

print("\n" + "=" * 80)
print("🎉 COMPLETE!")
print("=" * 80)
print(f"\n✅ Files saved:")
print(f"   1. {out_dir / 'results.json'} - Overall metrics")
print(f"   2. {out_dir / 'config.json'} - Configuration")
print(f"   3. {out_dir / 'linewise_cer_results.json'} - Detailed line-wise CER")
print(f"   4. {out_dir / 'linewise_cer_results.csv'} - CSV format")
print(f"   5. {out_dir / 'final_model'} - Trained model")
print(f"\n🔄 To run another configuration:")
print(f"   1. Go back to CELL 2")
print(f"   2. Change SELECTED_CONFIG values")
print(f"   3. Run CELL 2, then CELL 8, then CELL 9")
print("=" * 80)

# Clean up
del model
torch.cuda.empty_cache()


📊 COMPUTING LINE-WISE CER FOR EACH TEST SAMPLE

   Loading model from: /content/trocr_ablation_results/enc_3_dec_0_20260206_194608/final_model


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

   Test dataset size: 454 samples

🔍 Generating predictions for each test sample...


Processing: 100%|██████████| 454/454 [03:10<00:00,  2.38it/s]


📈 LINE-WISE CER STATISTICS:
Total test samples: 454
Average per-line CER: 0.0823
Overall CER (total edit distance / total chars): 0.0849
Min CER: 0.0000
Max CER: 0.6250
Median CER: 0.0625

✅ Line-wise CER results saved to:
   /content/trocr_ablation_results/enc_3_dec_0_20260206_194608/linewise_cer_results.json

✅ CSV file saved to:
   /content/trocr_ablation_results/enc_3_dec_0_20260206_194608/linewise_cer_results.csv

📝 SAMPLE PREDICTIONS (first 10):

Sample 1:
  Ground Truth: contar nol porea.
  Prediction:   contar no 'l porrea.
  CER:          0.1765

Sample 2:
  Ground Truth: Potrebbe t'aver per amança e tut
  Prediction:   Potrebbe t'aver per amança e tut
  CER:          0.0000

Sample 3:
  Ground Truth: ta sentir delectança chi ben ti
  Prediction:   t'asentir delectança chi benti
  CER:          0.0968

Sample 4:
  Ground Truth: portasse liança nel cuor sì come
  Prediction:   portasse liança nel cuor sì come
  CER:          0.0000

Sample 5:
  Ground Truth: dovrea.
  Predicti

In [ ]:
import shutil
from google.colab import files

model_path = '/content/trocr_ablation_results/enc_0_dec_0_20260206_165607/final_model'
output_filename = 'final_model_enc_0_dec_0'
shutil.make_archive(output_filename, 'zip', model_path)

print(f"\n✅ Model zipped to {output_filename}.zip. Initiating download...")
files.download(f'{output_filename}.zip')


✅ Model zipped to final_model_enc_0_dec_0.zip. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>